In [1]:
import customtkinter as ctk  
from tkinter import messagebox  
import firebase_admin
from firebase_admin import credentials, db
import json
from datetime import datetime

ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

class SistemaModerno:  
    def __init__(self):
        self.janela = ctk.CTk()
        self.janela.title("Sistema de Gerenciamento")
        self.janela.geometry("1200x800")
        
        if self.inicializar_firebase(): 
            self.criar_interface()
            # REMOVIDAS AS CHAMADAS DE CARREGAMENTO
            self.janela.mainloop()
        else:
            self.janela.destroy()
    
    def inicializar_firebase(self): 
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")  
                firebase_admin.initialize_app(cred, {
                    'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"  
                })
            
            self.db = db.reference()  
            print("Firebase Realtime Database conectado!")
            return True
            
        except FileNotFoundError:
            messagebox.showerror("Erro", "Arquivo 'bancochave.json' não encontrado!\n")
            return False
            
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha na conexão:\n{str(e)}")
            return False
    
    def criar_interface(self): # Design da Mari
        main_frame = ctk.CTkFrame(self.janela)
        main_frame.pack(fill="both", expand=True, padx=20, pady=20)
        
        ctk.CTkLabel(main_frame, text="Sistema de Gerenciamento", 
                    font=ctk.CTkFont(size=24, weight="bold")).pack(pady=20)
        
        frame_dados = ctk.CTkFrame(main_frame)
        frame_dados.pack(fill="both", expand=True)
        
        
        frame_usuarios = ctk.CTkFrame(frame_dados)
        frame_usuarios.pack(side="left", fill="both", expand=True, padx=10, pady=10)
        
        ctk.CTkLabel(frame_usuarios, text="Cadastro de Usuários", 
                    font=ctk.CTkFont(size=16, weight="bold")).pack(pady=10)
        
        ctk.CTkLabel(frame_usuarios, text="Nome:").pack(pady=(10, 5))
        self.entry_nome = ctk.CTkEntry(frame_usuarios, width=250)
        self.entry_nome.pack(pady=5)
        
        ctk.CTkLabel(frame_usuarios, text="Email:").pack(pady=(10, 5))
        self.entry_email = ctk.CTkEntry(frame_usuarios, width=250)
        self.entry_email.pack(pady=5)
        
        ctk.CTkLabel(frame_usuarios, text="Telefone:").pack(pady=(10, 5))
        self.entry_telefone = ctk.CTkEntry(frame_usuarios, width=250)
        self.entry_telefone.pack(pady=5)
        
        btn_usuario = ctk.CTkButton(frame_usuarios, text="Salvar Usuário", command=self.salvar_usuario, height=40)
        btn_usuario.pack(pady=20)
      
        
        
        frame_produtos = ctk.CTkFrame(frame_dados)
        frame_produtos.pack(side="right", fill="both", expand=True, padx=10, pady=10)
        
        ctk.CTkLabel(frame_produtos, text="Cadastro de Produtos", 
                    font=ctk.CTkFont(size=16, weight="bold")).pack(pady=10)
        
        campos_produto = [
            ("Nome do Produto:", "entry_nomeprod"),
            ("Quantidade:", "entry_quant_prod"),
            ("Descrição:", "entry_desc_prod"),
            ("Unidade (kg, un, pç):", "entry_uni_prod"),
            ("Fornecedor:", "entry_fornecedor"),
            ("Preço R$:", "entry_preco_prod")
        ]
        
        for label_text, attr_name in campos_produto:
            ctk.CTkLabel(frame_produtos, text=label_text).pack(pady=(5, 2))
            entry = ctk.CTkEntry(frame_produtos, width=250)
            entry.pack(pady=2)
            setattr(self, attr_name, entry)
        
        btn_produto = ctk.CTkButton(frame_produtos, text="Salvar Produto",
                                   command=self.cadastrar_produtos, height=40)
        btn_produto.pack(pady=20)
        
    
        
    
    def salvar_usuario(self):
        nome = self.entry_nome.get()
        email = self.entry_email.get()
        telefone = self.entry_telefone.get()
        
        if not nome or not email:
            messagebox.showwarning("Aviso", "Preencha nome e email!")
            return
        
        try:
            usuario_data = {
                'nome': nome,
                'email': email,
                'telefone': telefone,
                'data_cadastro': {'.sv': 'timestamp'}  
            }
            
            novo_usuario = self.db.child('usuarios').push(usuario_data)
            usuario_id = novo_usuario.key
            
            
            messagebox.showinfo("✅ Sucesso!", f"Usuário salvo com sucesso!\nID: {usuario_id}")
            
            
            self.entry_nome.delete(0, 'end')
            self.entry_email.delete(0, 'end')
            self.entry_telefone.delete(0, 'end')
            
            
        except Exception as e:
            messagebox.showerror("Erro", f"Falha ao salvar:\n{str(e)}")
    
    
    
    def cadastrar_produtos(self):
        nomeprod = self.entry_nomeprod.get()
        quant_prod = self.entry_quant_prod.get()
        desc_prod = self.entry_desc_prod.get()
        uni_prod = self.entry_uni_prod.get()
        fornecedor = self.entry_fornecedor.get()
        preco_prod = self.entry_preco_prod.get()

        if not nomeprod or not quant_prod:
            messagebox.showwarning("Aviso", "Preencha pelo menos Nome e Quantidade!")
            return
    
        try:
            quantidade = int(quant_prod)
            preco_str = preco_prod.replace(',', '.').strip()
            preco = float(preco_str) if preco_str else 0.0
        
            dados_produto = {
                "nome": nomeprod,
                "quantidade": quantidade,
                "descricao": desc_prod,
                "unidade": uni_prod.lower().strip(), 
                "fornecedor": fornecedor,
                "preco": preco,
                "data_cadastro": {'.sv': 'timestamp'},
                "ativo": True,
            }
        
            unidades_validas = ['kg', 'un', 'pc', 'pç', 'litro', 'm']
            if dados_produto['unidade'] not in unidades_validas:
                dados_produto['unidade'] = 'un'
                messagebox.showinfo("Info", f"Unidade '{uni_prod}' ajustada para 'un'")

            novo_produto = self.db.child('produtos').push(dados_produto)
            produto_id = novo_produto.key

            # Mensagem simplificada
            messagebox.showinfo("✅ Sucesso!", f"Produto salvo com sucesso!\nID: {produto_id}\nNome: {nomeprod}")
            
            # Limpar campos
            self.entry_nomeprod.delete(0, 'end')
            self.entry_quant_prod.delete(0, 'end')
            self.entry_desc_prod.delete(0, 'end')
            self.entry_uni_prod.delete(0, 'end')
            self.entry_fornecedor.delete(0, 'end')
            self.entry_preco_prod.delete(0, 'end')

            # REMOVIDA ATUALIZAÇÃO DE LISTA
            # self.carregar_produtos()

        except ValueError as ve:
            messagebox.showerror("Erro", f"Valor inválido: {str(ve)}\n\nPara preço, use: 12.50 ou 12,50")
        except Exception as e:
            messagebox.showerror("Erro", f"Falha ao salvar produto:\n{str(e)}")

if __name__ == "__main__":
    app = SistemaModerno()

Firebase Realtime Database conectado!


In [4]:
#Mari
import tkinter as tk
from tkinter import ttk, messagebox
import firebase_admin
from firebase_admin import credentials, db
from datetime import datetime

class SistemaEstoquePro:
    def __init__(self):
        self.janela = tk.Tk()
        self.janela.title("Cadastro de Produtos - Estoque Pro")
        self.configurar_janela()
        
        if self.inicializar_firebase():
            self.criar_interface()
            self.janela.mainloop()
        else:
            self.janela.destroy()
    
    def configurar_janela(self):
        """Configura o tamanho e posição da janela"""
        largura = 600
        altura = 350
        
        # Definir geometria
        self.janela.geometry(f"{largura}x{altura}")
        self.janela.configure(bg="#C2C2C2")
        
        # Centralizar na tela
        self.janela.update_idletasks()
        largura_tela = self.janela.winfo_screenwidth()
        altura_tela = self.janela.winfo_screenheight()
        posx = (largura_tela // 2) - (largura // 2)
        posy = (altura_tela // 2) - (altura // 2)
        self.janela.geometry(f"{largura}x{altura}+{posx}+{posy}")
        
        # Permitir expansão
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
    
    def inicializar_firebase(self):
        """Inicializa a conexão com o Firebase"""
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")  
                firebase_admin.initialize_app(cred, {
                    'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"  
                })
            
            self.db = db.reference()  
            print("✅ Firebase Realtime Database conectado!")
            return True
            
        except FileNotFoundError:
            messagebox.showerror("Erro", "Arquivo 'bancochave.json' não encontrado!\nColoque o arquivo na mesma pasta do programa.")
            return False
            
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha na conexão:\n{str(e)}")
            return False
    
    def criar_interface(self):
        """Cria a interface gráfica (DESIGNE ORIGINAL)"""
        
        # --- CABEÇALHO ---
        header = tk.Frame(self.janela, bg="#4A4AE8", height=100)
        header.grid(row=0, column=0, sticky="ew")  # Somente horizontal
        header.grid_propagate(False)  # Mantém altura fixa

        # Título principal (corrigido: estava duplicado)
        titulo_principal = tk.Label(
            header,
            text="Estoque Pro",
            bg="#4A4AE8",
            fg="white",
            font=("Arial", 16, "bold")
        )
        titulo_principal.pack(expand=True)

        # Subtítulo (corrigido: estava com typo "Contro toal")
        subtitulo = tk.Label(
            header,
            text="Controle total, resultado real",
            fg="white",
            bg="#4A4AE8",
            font=("Arial", 12, "bold")
        )
        subtitulo.pack(expand=True)

        # Estilo ttk
        style = ttk.Style()
        style.configure("TLabel", font=("Arial", 11), background="#F8F8F8")
        style.configure("TEntry", padding=3)
        style.configure("TButton", font=("Arial", 10, "bold"), padding=6)

        # Frame central
        frame = ttk.Frame(self.janela, padding=20)
        frame.grid(row=1, column=0)

        # Widgets - Criação dos campos
        self.labels = [
            ("Código do Produto:", "entry_codigo"),
            ("Nome do Produto:", "entry_nome"),
            ("Descrição do Produto:", "entry_descricao"),
            ("Unidade (un, kg, pç):", "entry_unidade"),
            ("Quantidade Mínima:", "entry_quantidade_min"),
            ("Fornecedor:", "entry_fornecedor")
        ]
        
        self.entries = {}
        
        for i, (texto, nome_var) in enumerate(self.labels):
            lbl = ttk.Label(frame, text=texto)
            lbl.grid(row=i, column=0, pady=5)

            ent = ttk.Entry(frame, width=40)
            ent.grid(row=i, column=1, pady=5)
            self.entries[nome_var] = ent

        # Frame dos botões
        botao_frame = ttk.Frame(frame, padding=10)
        botao_frame.grid(row=len(self.labels), column=0, columnspan=2, pady=(10, 0))

        # Botões com comandos
        ttk.Button(botao_frame, text="Salvar", command=self.salvar_produto).grid(row=0, column=0, padx=10)
        ttk.Button(botao_frame, text="Novo", command=self.limpar_campos).grid(row=0, column=1, padx=10)
        ttk.Button(botao_frame, text="Excluir", command=self.excluir_produto).grid(row=0, column=2, padx=10)
        
        # Adicionar botão de Listar (opcional)
        ttk.Button(botao_frame, text="Listar", command=self.listar_produtos).grid(row=0, column=3, padx=10)
    
    def salvar_produto(self):
        """Salva o produto no Firebase"""
        # Obter dados dos campos
        codigo = self.entries["entry_codigo"].get()
        nome = self.entries["entry_nome"].get()
        descricao = self.entries["entry_descricao"].get()
        unidade = self.entries["entry_unidade"].get()
        quantidade_min = self.entries["entry_quantidade_min"].get()
        fornecedor = self.entries["entry_fornecedor"].get()
        
        # Validação básica
        if not nome or not codigo:
            messagebox.showwarning("Aviso", "Preencha pelo menos Código e Nome do Produto!")
            return
        
        try:
            # Preparar dados para salvar
            dados_produto = {
                "codigo": codigo,
                "nome": nome,
                "descricao": descricao,
                "unidade": unidade.lower().strip(),
                "quantidade_minima": int(quantidade_min) if quantidade_min else 0,
                "fornecedor": fornecedor,
                "data_cadastro": {'.sv': 'timestamp'},
                "ativo": True
            }
            
            # Salvar no Firebase
            # Usar código como ID ou gerar automático
            produto_ref = self.db.child('produtos').child(codigo)
            produto_ref.set(dados_produto)
            
            messagebox.showinfo(
                "✅ Sucesso!", 
                f"Produto salvo com sucesso!\n\n"
                f"Código: {codigo}\n"
                f"Nome: {nome}\n"
                f"Fornecedor: {fornecedor}"
            )
            
            # Limpar campos após salvar
            self.limpar_campos()
            
        except ValueError as ve:
            messagebox.showerror("Erro", f"Valor inválido na quantidade: {str(ve)}")
        except Exception as e:
            messagebox.showerror("Erro", f"Falha ao salvar produto:\n{str(e)}")
    
    def limpar_campos(self):
        """Limpa todos os campos de entrada"""
        for entry in self.entries.values():
            entry.delete(0, tk.END)
    
    def excluir_produto(self):
        """Exclui um produto pelo código"""
        codigo = self.entries["entry_codigo"].get()
        
        if not codigo:
            messagebox.showwarning("Aviso", "Digite o código do produto para excluir!")
            return
        
        # Confirmar exclusão
        resposta = messagebox.askyesno(
            "Confirmar Exclusão", 
            f"Deseja realmente excluir o produto com código {codigo}?"
        )
        
        if resposta:
            try:
                self.db.child('produtos').child(codigo).delete()
                messagebox.showinfo("✅ Sucesso!", f"Produto {codigo} excluído com sucesso!")
                self.limpar_campos()
            except Exception as e:
                messagebox.showerror("Erro", f"Falha ao excluir produto:\n{str(e)}")
    
    def listar_produtos(self):
        """Lista todos os produtos cadastrados (em uma nova janela)"""
        try:
            produtos = self.db.child('produtos').get()
            
            if not produtos:
                messagebox.showinfo("Lista de Produtos", "Nenhum produto cadastrado.")
                return
            
            # Criar nova janela para listagem
            lista_window = tk.Toplevel(self.janela)
            lista_window.title("Lista de Produtos")
            lista_window.geometry("500x400")
            
            # Adicionar scrollbar
            scrollbar = tk.Scrollbar(lista_window)
            scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
            
            # Listbox para mostrar produtos
            listbox = tk.Listbox(lista_window, yscrollcommand=scrollbar.set, width=70, height=20)
            
            listbox.insert(tk.END, "📦 PRODUTOS CADASTRADOS:")
            listbox.insert(tk.END, "=" * 50)
            
            for produto_id, dados in produtos.items():
                listbox.insert(tk.END, f"\nCódigo: {produto_id}")
                listbox.insert(tk.END, f"Nome: {dados.get('nome', 'N/A')}")
                listbox.insert(tk.END, f"Fornecedor: {dados.get('fornecedor', 'N/A')}")
                listbox.insert(tk.END, f"Unidade: {dados.get('unidade', 'N/A')}")
                listbox.insert(tk.END, f"Descrição: {dados.get('descricao', 'N/A')[:30]}...")
                listbox.insert(tk.END, "-" * 40)
            
            listbox.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
            scrollbar.config(command=listbox.yview)
            
        except Exception as e:
            messagebox.showerror("Erro", f"Falha ao listar produtos:\n{str(e)}")

if __name__ == "__main__":
    app = SistemaEstoquePro()

✅ Firebase Realtime Database conectado!


In [4]:
#esterrrrr

import customtkinter as ctk
import tkinter as tk
from tkinter import messagebox
import firebase_admin 
from firebase_admin import credentials, db 
import json 

# Configuração de Tema do CustomTkinter
ctk.set_appearance_mode("light") 
ctk.set_default_color_theme("blue")

class LoginApp:
    def __init__(self):
        self.janela = ctk.CTk() 
        self.janela.title("Tela de Login - Estoque Pro")
        self.janela.geometry("500x350")
        self.janela.configure(fg_color="#F8F8F8")
        self.janela.iconbitmap('logoo-ofcc.ico')
        
        # Inicializar Firebase
        self.firebase_ok = self.inicializar_firebase()
        
        # Criar interface
        self.criar_interface()
        
        # Executar janela
        self.janela.mainloop()
    
    def inicializar_firebase(self): 
        """Inicializa a conexão com o Firebase"""
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")  
                firebase_admin.initialize_app(cred, {
                    'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"  
                })
            
            self.db = db.reference()  
            print("✅ Firebase Realtime Database conectado!")
            return True
            
        except FileNotFoundError:
            messagebox.showerror("Erro", "Arquivo 'bancochave.json' não encontrado!")
            return False
            
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha na conexão:\n{str(e)}")
            return False
    
    def criar_interface(self):
     
        frame_topo = ctk.CTkFrame(self.janela, fg_color="#236591", height=100, corner_radius=0)
        frame_topo.pack(fill="x", pady=0)
        
        # Título principal
        ctk.CTkLabel(frame_topo, 
                    text="Estoque Pro", 
                    fg_color="transparent", 
                    text_color="white", 
                    font=("Arial", 20, "bold")).pack(pady=(15, 5))
        
        # Subtítulo
        ctk.CTkLabel(frame_topo, 
                    text="Controle total, resultado real", 
                    fg_color="transparent", 
                    text_color="white", 
                    font=("Arial", 12)).pack(pady=(0, 15))
        
     
        frame_conteudo = ctk.CTkFrame(self.janela, fg_color="#F8F8F8") 
        frame_conteudo.pack(fill="both", expand=True, padx=20, pady=15)
        
        # Configurações de grid
        frame_conteudo.grid_rowconfigure(0, weight=1)
        frame_conteudo.grid_columnconfigure(0, weight=1)
        
      
        frame_azul_conteudo = ctk.CTkFrame(frame_conteudo, fg_color="#F8F8F8", corner_radius=10)
        frame_azul_conteudo.grid(row=0, column=0, padx=0, pady=0, sticky="nsew")
        
        # Configuração de expansão
        frame_azul_conteudo.grid_columnconfigure(0, weight=1) 
        frame_azul_conteudo.grid_columnconfigure(2, weight=1) 
        frame_azul_conteudo.grid_rowconfigure(0, weight=1) 
        frame_azul_conteudo.grid_rowconfigure(2, weight=1) 
        frame_azul_conteudo.grid_columnconfigure(1, weight=1) 
        frame_azul_conteudo.grid_rowconfigure(1, weight=1)
        
      
        frame_form = ctk.CTkFrame(frame_azul_conteudo, fg_color="#236591", corner_radius=10)
        frame_form.grid(row=1, column=1, sticky="nsew", padx=20, pady=20)
        
        # Configuração de expansão
        for i in range(6):
            frame_form.grid_rowconfigure(i, weight=1)
        frame_form.grid_columnconfigure(0, weight=1)
        frame_form.grid_columnconfigure(1, weight=1)
        frame_form.grid_columnconfigure(2, weight=1)
        
        
        # Título do formulário
        ctk.CTkLabel(frame_form, 
                     text="Faça seu Login", 
                     fg_color="transparent", 
                     text_color="white", 
                     font=("Arial", 26, "bold")).grid(row=0, column=0, columnspan=3, pady=(20, 10))
        
        # Usuário
        ctk.CTkLabel(frame_form, 
                     text="Usuário:", 
                     fg_color="transparent", 
                     text_color="white", 
                     font=("Arial", 13)).grid(row=1, column=0, padx=(20, 10), pady=10, sticky="e")
        
        self.entrada_usuario = ctk.CTkEntry(frame_form, 
                                          width=200, 
                                          height=35,
                                          fg_color="white", 
                                          text_color="#333", 
                                          border_width=1,
                                          font=("Arial", 12))
        self.entrada_usuario.grid(row=1, column=1, columnspan=2, padx=(0, 20), pady=10, sticky="w")
        
        # Senha
        ctk.CTkLabel(frame_form, 
                     text="Senha:", 
                     fg_color="transparent", 
                     text_color="white", 
                     font=("Arial", 13)).grid(row=2, column=0, padx=(20, 10), pady=10, sticky="e")
        
        self.entrada_senha = ctk.CTkEntry(frame_form, 
                                        width=200, 
                                        height=35,
                                        show="•", 
                                        fg_color="white", 
                                        text_color="#333", 
                                        border_width=1,
                                        font=("Arial", 12))
        self.entrada_senha.grid(row=2, column=1, columnspan=2, padx=(0, 20), pady=10, sticky="w")
        
        # Cargo
        ctk.CTkLabel(frame_form, 
                     text="Cargo:", 
                     fg_color="transparent", 
                     text_color="white", 
                     font=("Arial", 13)).grid(row=3, column=0, padx=(20, 10), pady=10, sticky="e")
        
        frame_radio = ctk.CTkFrame(frame_form, fg_color="transparent")
        frame_radio.grid(row=3, column=1, columnspan=2, padx=(0, 20), pady=10, sticky="w")
        
        self.cargo_selecionado = tk.StringVar(value="Funcionário")
        
        ctk.CTkRadioButton(frame_radio,
                          text="Funcionário",
                          variable=self.cargo_selecionado, 
                          value="Funcionário",
                          fg_color="white", 
                          border_color="white", 
                          hover_color="#4A8ABF", 
                          text_color="white", 
                          font=("Arial", 11)).pack(side="left", padx=(0, 15))
        
        ctk.CTkRadioButton(frame_radio,
                          text="Administrador",
                          variable=self.cargo_selecionado, 
                          value="Administrador",
                          fg_color="white", 
                          border_color="white", 
                          hover_color="#4A8ABF", 
                          text_color="white", 
                          font=("Arial", 11)).pack(side="left")
        
        # Botão Login
        btn_login = ctk.CTkButton(
            frame_form, 
            text="ENTRAR", 
            font=("Arial", 14, "bold"), 
            fg_color="white", 
            text_color="#236591", 
            hover_color="#F8F8F8",
            height=40,
            width=150,
            command=self.fazer_login
        )
        btn_login.grid(row=4, column=0, columnspan=3, padx=(0, 45))
        
        btn_cadastro = ctk.CTkButton(
            frame_form, 
            text="CADASTRAR", 
            font=("Arial", 14, "bold"), 
            fg_color="white", 
            text_color="#236591", 
            hover_color="#F8F8F8",
            height=40,
            width=150,
        )
        btn_cadastro.grid(row=4, column=0, columnspan=3, padx=(350, 45))
        
        # Status do Firebase
        status_color = "#6BFF70" if self.firebase_ok else "#440904"
        status_text = "✅ Conectado ao Banco de Dados" if self.firebase_ok else "❌ Banco de Dados Offline"
        
        status_label = ctk.CTkLabel(frame_form,
                                   text=status_text,
                                   fg_color="transparent",
                                   text_color=status_color,
                                   font=("Arial", 10))
        status_label.grid(row=5, column=0, columnspan=3, pady=(0, 10))
    
    def fazer_login(self):
        """Função de login que verifica credenciais no Firebase"""
        usuario = self.entrada_usuario.get().strip()
        senha = self.entrada_senha.get()
        cargo = self.cargo_selecionado.get()
        
        # Validação básica
        if not usuario or not senha:
            messagebox.showwarning("Erro de Login", "Preencha usuário e senha!")
            return
        
        if not self.firebase_ok:
            messagebox.showerror("Erro", "Banco de dados não conectado!")
            return
        
        try:
            # Buscar usuário no Firebase
            ref = self.db.child('usuarios')
            usuarios = ref.get()
            
            usuario_encontrado = None
            dados_usuario = None
            
            # Procurar usuário pelo nome ou email
            if usuarios:
                for user_id, user_data in usuarios.items():
                    if (user_data.get('nome') == usuario or 
                        user_data.get('email') == usuario or
                        user_data.get('usuario') == usuario):
                        usuario_encontrado = user_id
                        dados_usuario = user_data
                        break
            
            # Verificar credenciais
            if usuario_encontrado and dados_usuario:
                # Verificar senha (em produção, use hash!)
                if dados_usuario.get('senha') == senha:
                    # Verificar nível de acesso/cargo
                    nivel_usuario = dados_usuario.get('nivel_acesso', 'Funcionário')
                    
                    if nivel_usuario.lower() == cargo.lower() or nivel_usuario == "Administrador":
                        messagebox.showinfo("✅ Login Bem-sucedido", 
                                          f"Bem-vindo, {dados_usuario.get('nome', usuario)}!\n"
                                          f"Cargo: {nivel_usuario}")
                        
                        # Fechar janela de login (pode abrir próxima tela aqui)
                        self.abrir_sistema_principal(dados_usuario)
                    else:
                        messagebox.showerror("Acesso Negado", 
                                           f"Este usuário não tem permissão de {cargo}.\n"
                                           f"Nível de acesso: {nivel_usuario}")
                else:
                    messagebox.showerror("Erro de Login", "Senha incorreta!")
            else:
                messagebox.showerror("Erro de Login", "Usuário não encontrado!")
                
        except Exception as e:
            messagebox.showerror("Erro de Conexão", f"Falha ao consultar banco de dados:\n{str(e)}")
    
    def abrir_sistema_principal(self, dados_usuario):
        """Abre a tela principal do sistema após login bem-sucedido"""
        # Aqui você pode abrir a janela principal do sistema
        # Exemplo: SistemaModerno(dados_usuario)
        
        print(f"🚀 Iniciando sistema para: {dados_usuario.get('nome')}")
        print(f"📊 Nível de acesso: {dados_usuario.get('nivel_acesso')}")
        
        # Por enquanto, apenas fecha a janela de login
        # self.janela.destroy()
        
        # Para testar, mostra uma mensagem
        messagebox.showinfo("Sistema", f"Sistema principal será aberto para:\n"
                                      f"Nome: {dados_usuario.get('nome')}\n"
                                      f"Email: {dados_usuario.get('email')}\n"
                                      f"Nível: {dados_usuario.get('nivel_acesso')}")

if __name__ == "__main__":
    app = LoginApp()

✅ Firebase Realtime Database conectado!


In [6]:
#MARIOFCUSU

import tkinter as tk
from tkinter import ttk, messagebox
import firebase_admin
from firebase_admin import credentials, db
from datetime import datetime



def inicializar_firebase():
    try:
        if not firebase_admin._apps:
            cred = credentials.Certificate("bancochave.json")
            firebase_admin.initialize_app(cred, {
                'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"
            })

        print(" Firebase conectado com sucesso!")
        return db.reference("usuarios")   

    except FileNotFoundError:
        messagebox.showerror("Erro", "Arquivo 'bancochave.json' não encontrado!")
        return None

    except Exception as e:
        messagebox.showerror("Erro Firebase", f"Falha na conexão:\n{str(e)}")
        return None



janela = tk.Tk()
janela.title("Cadastro de Produtos - Estoque Pro")
janela.iconbitmap("logoo-ofcc.ico")
largura = 600
altura = 350
janela.geometry(f"{largura}x{altura}")
janela.configure(bg="#F8F8F8")


janela.update_idletasks()
largura_tela = janela.winfo_screenwidth()
altura_tela = janela.winfo_screenheight()
posx = (largura_tela // 2) - (largura // 2)
posy = (altura_tela // 2) - (altura // 2)
janela.geometry(f"{largura}x{altura}+{posx}+{posy}")


janela.grid_rowconfigure(1, weight=1)
janela.grid_columnconfigure(0, weight=1)

header = tk.Frame(janela, bg="#236591", height=100)

header.grid(row=0, column=0, sticky="ew")
header.grid_propagate(False)            
header.grid_columnconfigure(0, weight=1)
header.grid_columnconfigure(1, weight=1)

img_tk = tk.PhotoImage(file="logoo ofcc.png")
img_tk = img_tk.subsample(3)




titulo = tk.Label(header, text="Estoque Pro", bg="#236591",
                  fg="white", font=("Arial", 20, "bold"))
titulo.grid(row=0, column=1, sticky="w", padx=(0, 10), pady=(8, 0))

subtitulo = tk.Label(header, text="Controle total, resultado real",
                     fg="white", bg="#236591", font=("Arial", 12))
subtitulo.grid(row=1, column=1, sticky="w", padx=(0, 10), pady=(0, 8))



style = ttk.Style()
style.configure("TLabel", font=("Arial", 13), background="#F8F8F8")
style.configure("TEntry", padding=3)
style.configure("TButton", font=("Arial", 10, "bold"), padding=6)

frame = ttk.Frame(janela, padding=20)
frame.grid(row=1, column=0)

titulo_form = tk.Label(janela, text="Cadastro de Usuários",font=("Arial", 18, "bold"), fg="#236591", bg="#F8F8F8")
titulo_form.grid(row=1, column=0, pady=(0, 500)) 



labels = ["Nome:", "Usuário:", "Senha:", "Nível de Acesso:"]

entradas = {}  

for i, texto in enumerate(labels):
    lbl = ttk.Label(frame, text=texto)
    lbl.grid(row=i, column=0, sticky="e", pady=5, padx=10)

    ent = ttk.Entry(frame, width=40)
    ent.grid(row=i, column=1, pady=5, padx=10)

    entradas[texto[:-1].lower()] = ent 



def salvar_usuario():
    ref = inicializar_firebase()
    if ref is None:
        return

    dados = {
        "nome": entradas["nome"].get(),
        "usuario": entradas["usuário"].get(),
        "senha": entradas["senha"].get(),
        "nivel": entradas["nível de acesso"].get(),
        "data_cadastro": datetime.now().strftime("%d/%m/%Y %H:%M")
    }

    if not all(dados.values()):
        messagebox.showwarning("Aviso", "Preencha todos os campos.")
        return

    ref.push(dados)

    messagebox.showinfo("Sucesso", "Usuário cadastrado no Firebase!")

    for ent in entradas.values():
        ent.delete(0, tk.END)


botao_frame = ttk.Frame(frame, padding=10)
botao_frame.grid(row=len(labels), column=0, columnspan=2, pady=(10, 0))

ttk.Button(botao_frame, text="Salvar", command=salvar_usuario).grid(row=0, column=0, padx=10)
ttk.Button(botao_frame, text="Novo").grid(row=0, column=1, padx=10)
ttk.Button(botao_frame, text="Excluir").grid(row=0, column=2, padx=10)


janela.mainloop()


In [7]:
# cadprodofc - correção pack/grid e foco seguro

import tkinter as tk
from tkinter import ttk, messagebox
import firebase_admin
from firebase_admin import credentials, db
from datetime import datetime


class CadastroProdutosApp:
    def __init__(self):
        self.janela = tk.Tk()
        self.janela.title("Cadastro de Produtos - Estoque Pro")
        
        self.configurar_janela()
        
        self.db_ref = self.inicializar_firebase()
        
        self.criar_interface()
        
        self.janela.mainloop()
    
    def configurar_janela(self):
        largura = 600
        altura = 350
        
        self.janela.geometry(f"{largura}x{altura}")
        self.janela.configure(bg="#F8F8F8")
        self.janela.iconbitmap("logoo-ofcc.ico")
        
        # Centralizar
        self.janela.update_idletasks()
        largura_tela = self.janela.winfo_screenwidth()
        altura_tela = self.janela.winfo_screenheight()
        posx = (largura_tela // 2) - (largura // 2)
        posy = (altura_tela // 2) - (altura // 2)
        self.janela.geometry(f"{largura}x{altura}+{posx}+{posy}")
        
        self.janela.grid_rowconfigure(1, weight=1)
        self.janela.grid_columnconfigure(0, weight=1)
    
    def inicializar_firebase(self):
        """ Inicializa o Firebase de forma segura """
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")
                
                firebase_admin.initialize_app(cred, {
                    'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"
                })

            print("✅ Firebase conectado com sucesso!")
            return db.reference("produtos")

        except FileNotFoundError:
            messagebox.showerror(
                "Erro - Firebase",
                "O arquivo 'bancochave.json' NÃO FOI ENCONTRADO!\n"
                "Coloque ele na mesma pasta do script."
            )
            return None
        
        except Exception as e:
            messagebox.showerror(
                "Erro Firebase",
                f"Não foi possível conectar ao Firebase:\n\n{str(e)}"
            )
            return None
    
    def criar_interface(self):
        """Cria a interface gráfica"""
        # --- HEADER ---
        header = tk.Frame(self.janela, bg="#236591", height=100)
        header.grid(row=0, column=0, sticky="ew")

        header.grid_columnconfigure(0, weight=1)

        titulo_frame = tk.Frame(header, bg="#236591")
        titulo_frame.place(relx=0.5, rely=0.5, anchor="center")

        # Título
        titulo = tk.Label(
            titulo_frame,
            text="Estoque Pro",
            bg="#236591",
            fg="white",
            font=("Arial", 20, "bold")
        )
        titulo.pack(pady=(0, 0))

        # Subtítulo
        subtitulo = tk.Label(
            titulo_frame,
            text="Controle total, resultado real",
            fg="white",
            bg="#236591",
            font=("Arial", 12)
        )
        subtitulo.pack(pady=(0, 0))

        # --- MAIN ---
        main_frame = tk.Frame(self.janela, bg="#F8F8F8")
        main_frame.grid(row=1, column=0, sticky="nsew", padx=20, pady=(10, 20))

        titulo_form = tk.Label(
            main_frame,
            text="Cadastro de Produtos",
            bg="#F8F8F8",
            fg="#236591",
            font=("Arial", 18, "bold")
        )
        titulo_form.pack(pady=(0, 15))

        form_frame = tk.Frame(main_frame, bg="#F8F8F8")
        form_frame.pack()

        # Estilos
        style = ttk.Style()
        style.configure("TLabel", font=("Arial", 13), background="#F8F8F8")
        style.configure("TEntry", font=("Arial", 11), padding=3)
        style.configure("TButton", font=("Arial", 10, "bold"), padding=6)

        labels = [
            "Código do Produto:",
            "Nome do Produto:",
            "Descrição do Produto:",
            "Quantidade (un, kg, pç):",
            "Quantidade Mínima:",
            "Fornecedor:"
        ]

        self.entries = {}

        for i, texto in enumerate(labels):
            lbl = ttk.Label(form_frame, text=texto)
            lbl.grid(row=i, column=0, sticky="e", pady=5, padx=10)

            ent = ttk.Entry(form_frame, width=35)
            ent.grid(row=i, column=1, pady=5, padx=10)
            self.entries[texto] = ent

        # Botões
        botao_frame = tk.Frame(form_frame, bg="#F8F8F8")
        botao_frame.grid(row=len(labels), column=0, columnspan=2, pady=(20, 10))

        ttk.Button(botao_frame, text="Salvar", width=12, command=self.salvar_produto).grid(row=0, column=0, padx=10)
        ttk.Button(botao_frame, text="Novo", width=12, command=self.limpar_campos).grid(row=0, column=1, padx=10)
        ttk.Button(botao_frame, text="Sair", width=12, command=self.janela.destroy).grid(row=0, column=2, padx=10)

        # Status Firebase
        status_frame = tk.Frame(self.janela, bg="#E0E0E0", height=30)
        status_frame.grid(row=2, column=0, sticky="ew")

        status_text = "✅ Conectado ao Banco de Dados" if self.db_ref else "❌ Banco de Dados Offline"

        status_label = tk.Label(
            status_frame,
            text=status_text,
            bg="#E0E0E0",
            fg="#333",
            font=("Arial", 9)
        )
        status_label.pack(side="left", padx=10)

    def salvar_produto(self):
        """Salva o produto no Firebase"""

        if not self.db_ref:
            messagebox.showerror("Erro", "Firebase não está conectado!")
            return

        codigo = self.entries["Código do Produto:"].get().strip()
        nome = self.entries["Nome do Produto:"].get().strip()

        if not codigo or not nome:
            messagebox.showwarning("Aviso", "Código e Nome são obrigatórios!")
            return
        
        try:
            dados_produto = {
                "codigo": codigo,
                "nome": nome,
                "data_cadastro": {'.sv': 'timestamp'},
                "ativo": True
            }

            self.db_ref.push(dados_produto)

            messagebox.showinfo("Sucesso", "Produto cadastrado!")
            self.limpar_campos()

        except Exception as e:
            messagebox.showerror("Erro", f"Erro ao salvar produto:\n{e}")

    def limpar_campos(self):
        for entry in self.entries.values():
            entry.delete(0, tk.END)
        # foco seguro: só tenta se a chave existir
        if "Código do Produto:" in self.entries:
            try:
                self.entries["Código do Produto:"].focus_set()
            except Exception:
                pass


if __name__ == "__main__":
    app = CadastroProdutosApp()


✅ Firebase conectado com sucesso!


In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
import firebase_admin
from firebase_admin import credentials, db


# ============================
#  Inicializar Firebase
# ============================
def inicializar_firebase():
    try:
        if not firebase_admin._apps:
            cred = credentials.Certificate("bancochave.json")
            firebase_admin.initialize_app(cred, {
                'databaseURL': "https://bancodedadosprojeto-b4cec-default-rtdb.firebaseio.com/"
            })
        print("🔥 Firebase conectado com sucesso!")
        return db.reference("produtos")
    except Exception as e:
        messagebox.showerror("Erro ao conectar Firebase", str(e))
        return None


# ============================
#  JANELA PRINCIPAL
# ============================
janela = tk.Tk()
janela.title("Produtos Cadastrados")
janela.geometry("700x450")
janela.configure(bg="#C2C2C2")

janela.grid_rowconfigure(1, weight=1)
janela.grid_columnconfigure(0, weight=1)

# ============================
#  HEADER
# ============================
header = tk.Frame(janela, bg="#236591", height=120)
header.grid(row=0, column=0, sticky="ew")
header.grid_propagate(False)

header.grid_columnconfigure(0, weight=1)

titulo = tk.Label(
    header,
    text="Estoque Pro",
    bg="#236591",
    fg="white",
    font=("Arial", 22, "bold")
)
titulo.grid(row=0, column=0, pady=(15, 0))

subtitulo = tk.Label(
    header,
    text="Controle total, resultado real",
    bg="#236591",
    fg="white",
    font=("Arial", 12)
)
subtitulo.grid(row=1, column=0, pady=(0, 15))


# ============================
#  TABELA TREEVIEW
# ============================
colunas = ("Produto", "Quantidade")
tabela = ttk.Treeview(janela, columns=colunas, show="headings", height=10)

for col in colunas:
    tabela.heading(col, text=col)
    tabela.column(col, width=200)

tabela.grid(row=1, column=0, sticky="nsew", padx=10, pady=10)

scrollbar = ttk.Scrollbar(janela, orient="vertical", command=tabela.yview)
tabela.configure(yscroll=scrollbar.set)
scrollbar.grid(row=1, column=1, sticky="ns")


# ============================
#  FUNÇÃO PARA LISTAR PRODUTOS DO FIREBASE
# ============================
def carregar_produtos():
    tabela.delete(*tabela.get_children())
    ref = inicializar_firebase()
    dados = ref.get()

    if dados:
        for id_produto, item in dados.items():
            tabela.insert("", "end", iid=id_produto, values=(item.get("produto", "-"),
                                                             item.get("quantidade", 0)))
    else:
        messagebox.showinfo("Aviso", "Nenhum produto cadastrado.")


# ============================
#  FUNÇÕES DE ADICIONAR / REMOVER QUANTIDADE
# ============================
def alterar_quantidade(tipo):
    selecionado = tabela.focus()

    if not selecionado:
        messagebox.showwarning("Aviso", "Selecione um produto.")
        return

    ref = inicializar_firebase()
    produto_ref = ref.child(selecionado)
    produto = produto_ref.get()

    qtd_atual = produto["quantidade"]
    qtd = qtd_input.get()

    if not qtd.isdigit() or int(qtd) <= 0:
        messagebox.showwarning("Aviso", "Digite uma quantidade válida.")
        return

    qtd = int(qtd)

    if tipo == "remover":
        if qtd > qtd_atual:
            messagebox.showerror("Erro", "A quantidade a remover é maior que o estoque!")
            return
        nova_qtd = qtd_atual - qtd
    else:
        nova_qtd = qtd_atual + qtd

    produto_ref.update({"quantidade": nova_qtd})
    carregar_produtos()
    qtd_input.delete(0, tk.END)


# ============================
#  CONTROLES
# ============================
controle_frame = tk.Frame(janela, bg="#C2C2C2")
controle_frame.grid(row=2, column=0, pady=10)

tk.Label(controle_frame, text="Quantidade:", bg="#C2C2C2").grid(row=0, column=0, padx=5)
qtd_input = tk.Entry(controle_frame, width=10)
qtd_input.grid(row=0, column=1, padx=5)

btn_remover = tk.Button(controle_frame, text="Remover", bg="#D9534F", fg="white",
                        command=lambda: alterar_quantidade("remover"))
btn_remover.grid(row=0, column=2, padx=10)

btn_adicionar = tk.Button(controle_frame, text="Adicionar", bg="#5CB85C", fg="white",
                          command=lambda: alterar_quantidade("adicionar"))
btn_adicionar.grid(row=0, column=3, padx=10)

btn_atualizar = tk.Button(controle_frame, text="Atualizar Tabela", command=carregar_produtos)
btn_atualizar.grid(row=0, column=4, padx=10)


# ============================
# INICIAR
# ============================
janela.mainloop()


🔥 Firebase conectado com sucesso!


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\ADM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\firebase_admin\db.py", line 928, in request
    return super().request(method, url, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ADM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\firebase_admin\_http_client.py", line 136, in request
    resp.raise_for_status()
  File "C:\Users\ADM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: https://bancodedadosprojeto-b4cec-default-rtdb.firebaseio.com/produtos.json

During handling of the 

In [4]:
import tkinter as tk
from tkinter import ttk, messagebox
import firebase_admin
from firebase_admin import credentials, db


# ============================================
# 🔥 INICIALIZAR FIREBASE (somente 1 vez)
# ============================================
def inicializar_firebase():
    try:
        if not firebase_admin._apps:  
            cred = credentials.Certificate("bancochave.json")  # ← sua chave
            firebase_admin.initialize_app(cred, {
                'databaseURL': "https://bancodedadosprojeto-b4cec-default-rtdb.firebaseio.com/"
            })

        print("🔥 Firebase conectado!")
        return db.reference("produtos")

    except Exception as e:
        messagebox.showerror("Erro ao conectar Firebase", str(e))
        return None


# ============================================
#  🪟 JANELA PRINCIPAL
# ============================================
janela = tk.Tk()
janela.title("Produtos Cadastrados")
janela.geometry("700x450")
janela.configure(bg="#C2C2C2")

janela.grid_rowconfigure(1, weight=1)
janela.grid_columnconfigure(0, weight=1)

# ============================================
#  HEADER
# ============================================
header = tk.Frame(janela, bg="#236591", height=120)
header.grid(row=0, column=0, sticky="ew")
header.grid_propagate(False)

titulo = tk.Label(
    header,
    text="Estoque Pro",
    bg="#236591",
    fg="white",
    font=("Arial", 22, "bold")
)
titulo.grid(row=0, column=0, pady=(15, 0))

subtitulo = tk.Label(
    header,
    text="Controle total, resultado real",
    bg="#236591",
    fg="white",
    font=("Arial", 12)
)
subtitulo.grid(row=1, column=0, pady=(0, 15))


# ============================================
#  📋 TABELA (Treeview)
# ============================================
colunas = ("Produto", "Quantidade")
tabela = ttk.Treeview(janela, columns=colunas, show="headings", height=10)

for col in colunas:
    tabela.heading(col, text=col)
    tabela.column(col, width=200)

tabela.grid(row=1, column=0, sticky="nsew", padx=10, pady=10)

scrollbar = ttk.Scrollbar(janela, orient="vertical", command=tabela.yview)
tabela.configure(yscroll=scrollbar.set)
scrollbar.grid(row=1, column=1, sticky="ns")


# ============================================
#  🔄 CARREGAR PRODUTOS DO FIREBASE
# ============================================
def carregar_produtos():
    tabela.delete(*tabela.get_children())

    ref = inicializar_firebase()
    if not ref:
        return

    dados = ref.get()

    if dados:
        for id_produto, item in dados.items():
            tabela.insert("", "end",
                          iid=id_produto,
                          values=(item.get("produto", "-"),
                                  item.get("quantidade", 0)))
    else:
        messagebox.showinfo("Aviso", "Nenhum produto cadastrado.")


# ============================================
#  ➕➖ ADICIONAR / REMOVER QUANTIDADE
# ============================================
def alterar_quantidade(tipo):
    selecionado = tabela.focus()

    if not selecionado:
        messagebox.showwarning("Aviso", "Selecione um produto.")
        return

    ref = inicializar_firebase()
    produto_ref = ref.child(selecionado)
    produto = produto_ref.get()

    qtd_atual = int(produto["quantidade"])
    qtd = qtd_input.get()

    if not qtd.isdigit() or int(qtd) <= 0:
        messagebox.showwarning("Aviso", "Digite uma quantidade válida.")
        return

    qtd = int(qtd)

    if tipo == "remover":
        if qtd > qtd_atual:
            messagebox.showerror("Erro", "A quantidade a remover é maior que o estoque!")
            return
        nova_qtd = qtd_atual - qtd
    else:
        nova_qtd = qtd_atual + qtd

    produto_ref.update({"quantidade": nova_qtd})
    carregar_produtos()
    qtd_input.delete(0, tk.END)


# ============================================
#  🎛️ CONTROLES
# ============================================
controle_frame = tk.Frame(janela, bg="#C2C2C2")
controle_frame.grid(row=2, column=0, pady=10)

tk.Label(controle_frame, text="Quantidade:", bg="#C2C2C2").grid(row=0, column=0, padx=5)

qtd_input = tk.Entry(controle_frame, width=10)
qtd_input.grid(row=0, column=1, padx=5)

btn_remover = tk.Button(controle_frame, text="Remover", bg="#D9534F", fg="white",
                        command=lambda: alterar_quantidade("remover"))
btn_remover.grid(row=0, column=2, padx=10)

btn_adicionar = tk.Button(controle_frame, text="Adicionar", bg="#5CB85C", fg="white",
                          command=lambda: alterar_quantidade("adicionar"))
btn_adicionar.grid(row=0, column=3, padx=10)

btn_atualizar = tk.Button(controle_frame, text="Atualizar Tabela", command=carregar_produtos)
btn_atualizar.grid(row=0, column=4, padx=10)

# ============================================
#  🚀 INICIAR
# ============================================
janela.mainloop()


In [6]:
import tkinter as tk
from tkinter import ttk, messagebox
import firebase_admin
from firebase_admin import credentials, db
from datetime import datetime


# ============================================
# 🔥 INICIALIZAR FIREBASE
# ============================================
def inicializar_firebase():
    try:
        if not firebase_admin._apps:
            cred = credentials.Certificate("bancochave.json")
            firebase_admin.initialize_app(cred, {
                "databaseURL": "https://bancodedadosprojeto-b4cec-default-rtdb.firebaseio.com/"
            })
        return db.reference("produtos")
    except Exception as e:
        messagebox.showerror("Erro ao conectar Firebase", str(e))
        return None


# ============================================
# 🪟 JANELA PRINCIPAL
# ============================================
janela = tk.Tk()
janela.title("Produtos com Estoque Baixo")
janela.geometry("750x500")
janela.configure(bg="#C2C2C2")

janela.grid_rowconfigure(1, weight=1)
janela.grid_columnconfigure(0, weight=1)

# ============================================
# HEADER
# ============================================
header = tk.Frame(janela, bg="#9A161F", height=120)
header.grid(row=0, column=0, sticky="ew")
header.grid_propagate(False)

titulo = tk.Label(
    header, text="Estoque Pro",
    bg="#9A161F", fg="white",
    font=("Arial", 22, "bold")
)
titulo.grid(row=0, column=0, pady=(15, 0))

subtitulo = tk.Label(
    header, text="Produtos com Estoque Baixo",
    bg="#9A161F", fg="white",
    font=("Arial", 14)
)
subtitulo.grid(row=1, column=0, pady=(0, 15))


# ============================================
# 📋 TABELA
# ============================================
colunas = ("Produto", "Quantidade", "Qtd. Mínima")
tabela = ttk.Treeview(janela, columns=colunas, show="headings", height=10)

for col in colunas:
    tabela.heading(col, text=col)
    tabela.column(col, width=200)

tabela.grid(row=1, column=0, sticky="nsew", padx=10, pady=10)

scroll = ttk.Scrollbar(janela, orient="vertical", command=tabela.yview)
tabela.configure(yscroll=scroll.set)
scroll.grid(row=1, column=1, sticky="ns")


# ============================================
# 🔄 CARREGAR PRODUTOS COM ESTOQUE BAIXO
# ============================================
def carregar_baixo_estoque():
    tabela.delete(*tabela.get_children())

    ref = inicializar_firebase()
    dados = ref.get()

    if not dados:
        messagebox.showinfo("Aviso", "Nenhum produto encontrado.")
        return

    for idp, item in dados.items():
        qtd = int(item.get("quantidade", 0))
        minimo = int(item.get("quantidadeMinima", 0))

        if qtd <= minimo:  # 🔥 produto com estoque baixo
            tabela.insert("", "end", iid=idp,
                          values=(item.get("produto", "-"), qtd, minimo))


# ============================================
# ➕ Registrar motivo do estoque baixo
# ============================================
def registrar_motivo():
    selecionado = tabela.focus()

    if not selecionado:
        messagebox.showwarning("Aviso", "Selecione um produto.")
        return

    motivo = combo_motivo.get()
    if motivo == "":
        messagebox.showwarning("Aviso", "Selecione um motivo.")
        return

    ref_base = inicializar_firebase()
    produto = ref_base.child(selecionado).get()
    qtd_atual = int(produto["quantidade"])

    # salva movimentação
    movimentos_ref = db.reference(f"movimentacoes/{selecionado}")
    movimentos_ref.push({
        "motivo": motivo,
        "quantidade": qtd_atual,
        "data": datetime.now().strftime("%d/%m/%Y %H:%M")
    })

    messagebox.showinfo("Registrado",
                        f"Motivo '{motivo}' registrado para o produto '{produto['produto']}'.")
    combo_motivo.set("")


# ============================================
# 🎛️ CONTROLES
# ============================================
controle = tk.Frame(janela, bg="#C2C2C2")
controle.grid(row=2, column=0, pady=10)

tk.Label(controle, text="Motivo:", bg="#C2C2C2").grid(row=0, column=0, padx=5)

combo_motivo = ttk.Combobox(
    controle,
    values=["Retirada", "Perda", "Vencimento", "Ajuste Manual", "Outro"],
    width=20,
    state="readonly"
)
combo_motivo.grid(row=0, column=1, padx=5)

btn_registrar = tk.Button(
    controle,
    text="Registrar Motivo",
    bg="#9A161F", fg="white",
    command=registrar_motivo
)
btn_registrar.grid(row=0, column=2, padx=10)

btn_recarregar = tk.Button(controle, text="Atualizar Lista", command=carregar_baixo_estoque)
btn_recarregar.grid(row=0, column=3, padx=10)


# ============================================
# 🚀 INICIAR
# ============================================
janela.mainloop()


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\ADM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\firebase_admin\db.py", line 928, in request
    return super().request(method, url, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ADM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\firebase_admin\_http_client.py", line 136, in request
    resp.raise_for_status()
  File "C:\Users\ADM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: https://bancodedadosprojeto-b4cec-default-rtdb.firebaseio.com/produtos.json

During handling of the 

In [8]:
import tkinter as tk
from tkinter import ttk
import firebase_admin 
from firebase_admin import credentials, db 
import json 
from datetime import datetime


def inicializar_firebase(self): 
        """Inicializa a conexão com o Firebase"""
        try:
            if not firebase_admin._apps:
                cred = credentials.Certificate("bancochave.json")  
                firebase_admin.initialize_app(cred, {
                    'databaseURL': "https://bancoback-3c307-default-rtdb.firebaseio.com/"  
                })
            
            self.db = db.reference()  
            print("✅ Firebase Realtime Database conectado!")
            return True
            
        except FileNotFoundError:
            messagebox.showerror("Erro", "Arquivo 'bancochave.json' não encontrado!")
            return False
            
        except Exception as e:
            messagebox.showerror("Erro Firebase", f"Falha na conexão:\n{str(e)}")
            return False
    
janela = tk.Tk()
janela.title("Cadastro de Produtos")
janela.iconbitmap("logoo-ofcc.ico")
largura = 600
altura = 350
janela.geometry(f"{largura}x{altura}")
janela.configure(bg="#C2C2C2")

# ----- Centralizar a janela -----
janela.update_idletasks()
largura_tela = janela.winfo_screenwidth()
altura_tela = janela.winfo_screenheight()
posx = (largura_tela // 2) - (largura // 2)
posy = (altura_tela // 2) - (altura // 2)
janela.geometry(f"{largura}x{altura}+{posx}+{posy}")

# Permitir expansão
janela.grid_rowconfigure(1, weight=1)
janela.grid_columnconfigure(0, weight=1)

# --- CABEÇALHO ---
header = tk.Frame(janela, bg="#236591", height=150)
# colocar o header na grid da janela (linha 0)
header.grid(row=0, column=0, sticky="ew")
header.grid_propagate(False)             # mantém a altura definida
header.grid_columnconfigure(0, weight=1)
header.grid_columnconfigure(1, weight=1)

img_tk = tk.PhotoImage(file="logoo ofcc.png")
img_tk = img_tk.subsample(3)

labels.entry_nome = ctk.CTkEntry(labels, width=250)
labels.entry_nome.pack(pady=5)
self.entry_usuario= ctk.CTkEntry(labels, width=250)
self.entry_usuario.pack(pady=5)
self.entry_senha= ctk.CTkEntry(labels, width=250)
self.entry_senha.pack(pady=5)
self.entry_niveldeacesso= ctk.CTkEntry(labels, width=250)
self.entry_niveldeacesso.pack(pady=5)
# Ícone no cabeçalho
icone_label = tk.Label(header, image=img_tk, bg="#236591")
icone_label.grid(row=0, column=0, rowspan=2, padx=(0, 10), pady=(8, 0))

# Título
titulo = tk.Label(
    header,
    text="Estoque Pro",
    bg="#236591",
    fg="white",
    font=("Arial", 20, "bold")
)
titulo.grid(row=0, column=1, sticky="w", padx=(0, 10), pady=(8, 0))

# Subtítulo (aparece embaixo do título)
subtitulo = tk.Label(
    header,
    text="Controle total, resultado real",
    fg="white",
    bg="#236591",
    font=("Arial", 12)
)
subtitulo.grid(row=1, column=1, sticky="w", padx=(0, 10), pady=(0, 8))


# Estilo ttk
style = ttk.Style()
style.configure("TLabel", font=("Arial", 13), background="#F8F8F8")
style.configure("TEntry", padding=3)
style.configure("TButton", font=("Arial", 10, "bold"), padding=6)

frame = ttk.Frame(janela, padding=20)
frame.grid(row=1, column=0)


titulo_form = tk.Label(janela, text="Cadastro de Usuários",font=("Arial", 18, "bold"))
titulo_form.grid(row=1, column=0, pady=(0, 500))  # espaço em cima e embaixo

# Widgets
labels = [
    "Nome:",
    "Usuário:",
    "Senha:",
    "Nível de Acesso:"
]
entries = ["entry_nome"]

for i, texto in enumerate(labels):
    lbl = ttk.Label(frame, text=texto)
    lbl.grid(row=i, column=0, sticky="e", pady=5, padx=10)

    ent = ttk.Entry(frame, width=40)
    ent.grid(row=i, column=1, pady=5, padx=10)
    entries.append(ent)

# Frame dos botões (dentro do card)
botao_frame = ttk.Frame(frame, padding=10)
botao_frame.grid(row=len(labels), column=0, columnspan=2, pady=(10, 0))

ttk.Button(botao_frame, text="Salvar").grid(row=0, column=0, padx=10)
ttk.Button(botao_frame, text="Novo").grid(row=0, column=1, padx=10)
ttk.Button(botao_frame, text="Excluir").grid(row=0, column=2, padx=10)

janela.mainloop()


AttributeError: 'list' object has no attribute 'tk'

In [ ]:
import tkinter as tk
from tkinter import ttk
import firebase_admin
from firebase_admin import credentials, db

# ---------------- Firebase Admin Config ----------------
cred = credentials.Certificate("bancochave.json")

firebase_admin.initialize_app(cred, {
    "databaseURL": "https://bancodedadosprojeto-b4cec-default-rtdb.firebaseio.com"
})
# ------------------------------------------------------


# Janela Tkinter
root = tk.Tk()
root.title("Estoque Pro - Tkinter Firebase Admin")
root.geometry("900x600")


# -------- Título Geral --------
label_geral = tk.Label(root, text="Relatório Geral", font=("Arial", 18, "bold"))
label_geral.pack(pady=5)

cols = ("Produto", "Descrição", "Quantidade", "Cadastro", "Situação")
tree = ttk.Treeview(root, columns=cols, show="headings", height=10)

for col in cols:
    tree.heading(col, text=col)
    tree.column(col, width=150)

tree.pack(pady=5)


# -------- Relatório Individual --------
label_ind = tk.Label(root, text="Relatório Individual", font=("Arial", 18, "bold"))
label_ind.pack(pady=10)

produtos_cb = ttk.Combobox(root, width=40, state="readonly")
produtos_cb.pack()

frame_info = tk.Frame(root)
frame_info.pack(pady=10)

lbl_nome = tk.Label(frame_info, text="", font=("Arial", 16, "bold"))
lbl_desc = tk.Label(frame_info, text="")
lbl_qtd = tk.Label(frame_info, text="")
lbl_data = tk.Label(frame_info, text="")

lbl_nome.pack()
lbl_desc.pack()
lbl_qtd.pack()
lbl_data.pack()


# -------- Função Relatório Geral --------
def carregar_relatorio_geral():
    tree.delete(*tree.get_children())
    produtos_cb["values"] = []

    ref = db.reference("produtos")
    dados = ref.get()

    produtos = []

    if dados:
        for id, item in dados.items():

            quantidade = int(item.get("quantidade", 0))
            if quantidade == 0:
                situacao = "Zerado"
            elif quantidade <= 5:
                situacao = "Baixo estoque"
            else:
                situacao = "OK"

            tree.insert("", "end", values=(
                item.get("produto"),
                item.get("descricao"),
                quantidade,
                item.get("data"),
                situacao
            ))

            produtos.append(item.get("produto"))

        produtos_cb["values"] = produtos


# -------- Função Relatório Individual --------
def carregar_individual(event):
    produto_nome = produtos_cb.get()

    ref = db.reference("produtos")
    dados = ref.get()

    if dados:
        for id, item in dados.items():
            if item.get("produto") == produto_nome:
                lbl_nome.config(text=item.get("produto"))
                lbl_desc.config(text="Descrição: " + item.get("descricao", ""))
                lbl_qtd.config(text="Quantidade: " + str(item.get("quantidade")))
                lbl_data.config(text="Cadastro: " + item.get("data"))
                break


produtos_cb.bind("<<ComboboxSelected>>", carregar_individual)

# Carregar dados na inicialização
carregar_relatorio_geral()

root.mainloop()

ValueError: The default Firebase app already exists. This means you called initialize_app() more than once without providing an app name as the second argument. In most cases you only need to call initialize_app() once. But if you do want to initialize multiple apps, pass a second argument to initialize_app() to give each app a unique name.

In [ ]:
import customtkinter as ctk
import tkinter as tk
from tkinter import messagebox
import firebase_admin 
from firebase_admin import credentials, db 
import json 

janela = tk.Tk()
janela.title("Cadastro de Produtos - Estoque Pro")

janela.geometry('600x350')
janela.configure(bg="#F8F8F8")
janela.iconbitmap("logoo-ofcc.ico")

janela.grid_rowconfigure(1, weight=1)
janela.grid_columnconfigure(0, weight=1)

# ================= HEADER ====================
header = tk.Frame(janela, bg="#236591", height=100)
header.grid(row=0, column=0, sticky="ew")
header.grid_columnconfigure(0, weight=1)

titulo_frame = tk.Frame(header, bg="#236591")
titulo_frame.place(relx=0.5, rely=0.5, anchor="center")

titulo = tk.Label(
    titulo_frame,
    text="Estoque Pro",
    bg="#236591",
    fg="white",
    font=("Arial", 20, "bold")
)
titulo.pack()

subtitulo = tk.Label(
    titulo_frame,
    text="Controle total, resultado real",
    fg="white",
    bg="#236591",
    font=("Arial", 12)
)
subtitulo.pack()

# ================= MAIN ======================
main_frame = tk.Frame(janela, bg="#F8F8F8")
main_frame.grid(row=1, column=0, sticky="nsew", padx=20, pady=(10, 20))

titulo_form = tk.Label(
    main_frame,
    text="Tela de Acesso - ADMINISTRADOR",
    bg="#F8F8F8",
    fg="#236591",
    font=("Arial", 18, "bold")
)
titulo_form.pack(pady=(0, 15))

form_frame = tk.Frame(main_frame, bg="#F8F8F8")
form_frame.pack()

# ================= BOTÕES LADO A LADO ======================
botoes_frame = tk.Frame(main_frame, bg="#F8F8F8")
botoes_frame.pack(pady=10)

btn1 = tk.Button(botoes_frame, text="Cadastro de Produtos", width=20, bg="#236591", height=5, fg="white", font=("Arial", 15))
btn1.grid(row=0, column=0, padx=5)

btn2 = tk.Button(botoes_frame, text="Entrada e Saída", width=20, bg="#236591", height=5, fg="white", font=("Arial", 15))
btn2.grid(row=0, column=1, padx=5)

btn3 = tk.Button(botoes_frame, text="Movimentações Registradas", width=25, bg="#236591", height=5, fg="white", font=("Arial", 15))
btn3.grid(row=0, column=2, padx=20)

btn4 = tk.Button(botoes_frame, text="Dashboard", width=20, bg="#236591", height=5, fg="white", font=("Arial", 15))
btn4.grid(row=2, column=0, padx=20, pady=10)

btn5 = tk.Button(botoes_frame, text="Relatório", width=20, bg="#236591", height=5, fg="white", font=("Arial", 15))
btn5.grid(row=2, column=1, padx=5)


# =========================================================

janela.mainloop()


TclError: bitmap "logoo-ofcc.ico" not defined

In [26]:
import customtkinter as ctk
import tkinter as tk
from tkinter import messagebox
import firebase_admin 
from firebase_admin import credentials, db 
import json 

janela = tk.Tk()
janela.title("Cadastro de Produtos - Estoque Pro")

janela.geometry('600x350')
janela.configure(bg="#F8F8F8")
janela.iconbitmap("logoo-ofcc.ico")

janela.grid_rowconfigure(1, weight=1)
janela.grid_columnconfigure(0, weight=1)

# ================= HEADER ====================
header = tk.Frame(janela, bg="#236591", height=100)
header.grid(row=0, column=0, sticky="ew")
header.grid_columnconfigure(0, weight=1)

titulo_frame = tk.Frame(header, bg="#236591")
titulo_frame.place(relx=0.5, rely=0.5, anchor="center")

titulo = tk.Label(
    titulo_frame,
    text="Estoque Pro",
    bg="#236591",
    fg="white",
    font=("Arial", 20, "bold")
)
titulo.pack()

subtitulo = tk.Label(
    titulo_frame,
    text="Controle total, resultado real",
    fg="white",
    bg="#236591",
    font=("Arial", 12)
)
subtitulo.pack()

# ================= MAIN ======================
main_frame = tk.Frame(janela, bg="#F8F8F8")
main_frame.grid(row=1, column=0, sticky="nsew", padx=20, pady=(10, 20))

titulo_form = tk.Label(
    main_frame,
    text="Tela de Acesso - ADMINISTRADOR",
    bg="#F8F8F8",
    fg="#236591",
    font=("Arial", 18, "bold")
)
titulo_form.pack(pady=(0, 15))

form_frame = tk.Frame(main_frame, bg="#F8F8F8")
form_frame.pack()

janela.mainloop()



In [4]:
import tkinter as tk
import subprocess
import os

def abrir_excel():
    # coloque aqui o caminho do arquivo que você quer abrir
    caminho = r"Relatório Estoque-Pro.xlsx"
    
    if os.path.exists(caminho):
        subprocess.Popen([caminho], shell=True)
    else:
        print("Arquivo não encontrado:", caminho)

janela = tk.Tk()
janela.title("Abrir Excel")

botao = tk.Button(janela, text="Abrir Arquivo Excel", command=abrir_excel)
botao.pack(padx=20, pady=20)

janela.mainloop()


In [8]:
import tkinter as tk

class App(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("Juntar Dois Códigos Tkinter")
        self.geometry("600x400")

        container = tk.Frame(self)
        container.pack(fill="both", expand=True)

        # Armazena as páginas
        self.paginas = {}

        # Aqui você registra todas as páginas que existirão no app
        for P in (Pagina1, Pagina2):
            pagina = P(container, self)
            self.paginas[P] = pagina
            pagina.grid(row=0, column=0, sticky="nsew")

        # Mostra a primeira tela
        self.mostrar_pagina(Pagina1)

    def mostrar_pagina(self, pagina):
        frame = self.paginas[pagina]
        frame.tkraise()


# =======================
# PÁGINA 1 (COLE SEU CÓDIGO AQUI)
# =======================
class Pagina1(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent)

        tk.Label(self, text="ESTE É O CÓDIGO 1",
                 font=("Arial", 20)).pack(pady=20)

        # ---> Botão que abre a página 2
        tk.Button(self,
                  text="IR PARA O CÓDIGO 2",
                  command=lambda: controller.mostrar_pagina(Pagina2)
                ).pack(pady=10)
        
class Pagina2(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent)

        tk.Label(self, text="ESTE É O CÓDIGO 2",
                 font=("Arial", 20)).pack(pady=20)

        # ---> Botão que volta para a página 1
        tk.Button(self,
                  text="VOLTAR PARA O CÓDIGO 1",
                  command=lambda: controller.mostrar_pagina(Pagina1)
                ).pack(pady=10)


# =======================
# INICIAR O APP
# =======================
App().mainloop()




In [9]:
import tkinter as tk

class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Links no Tkinter")
        self.geometry("400x300")

        container = tk.Frame(self)
        container.pack(fill="both", expand=True)

        self.paginas = {}

        for P in (Pagina1, Pagina2):
            pagina = P(container, self)
            self.paginas[P] = pagina
            pagina.grid(row=0, column=0, sticky="nsew")

        self.mostrar_pagina(Pagina1)

    def mostrar_pagina(self, pagina):
        self.paginas[pagina].tkraise()


class Pagina1(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent)

        tk.Label(self, text="Página 1", font=("Arial", 20)).pack(pady=20)

        link = tk.Label(self, text="Ir para página 2",
                        fg="blue", cursor="hand2", font=("Arial", 12, "underline"))
        link.pack()

        link.bind("<Button-1>", lambda e: controller.mostrar_pagina(Pagina2))


class Pagina2(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent)

        tk.Label(self, text="Página 2", font=("Arial", 20)).pack(pady=20)

        link = tk.Label(self, text="Voltar para página 1",
                        fg="blue", cursor="hand2", font=("Arial", 12, "underline"))
        link.pack()

        link.bind("<Button-1>", lambda e: controller.mostrar_pagina(Pagina1))


App().mainloop()
